# Assignment 1 – Fixed-Size Chunking, Embeddings, and Cosine Similarity

This notebook uses **world_foods.txt**. It implements fixed-size character chunking with 15% overlap, SentenceTransformer embeddings, cosine similarity, and Top-3 retrieval.


In [1]:
!pip -q install sentence-transformers scikit-learn


## 1. Load the text file
Upload `world_foods.txt` when running in Google Colab.


In [14]:
file_path = '/content/world_foods.txt'

with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

print('File loaded successfully.')
print('Total characters:', len(text))
print('\nPreview:\n', text[:500])


File loaded successfully.
Total characters: 2001

Preview:
 Sushi is a Japanese dish made of vinegared rice combined with fresh seafood, vegetables, or egg. It is often served with soy sauce, wasabi, and pickled ginger. Nigiri, maki, and sashimi are the most common styles.

Pizza Margherita comes from Naples, Italy. It is topped with tomato sauce, fresh mozzarella, basil, and olive oil. The colors of the toppings are said to represent the Italian flag.

Tacos are a traditional Mexican food made of a small corn or wheat tortilla folded around a filling. P


## 2. Fixed-Size Window Chunking with 15% Overlap
Chunk size = 500 characters. Overlap = 75 characters (15%).


In [16]:
CHUNK_SIZE = 400
OVERLAP = int(CHUNK_SIZE * 0.15)
STEP = CHUNK_SIZE - OVERLAP

def fixed_size_chunking(text, chunk_size=500, overlap=75):
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(text):
            break
    return chunks

chunks = fixed_size_chunking(text, CHUNK_SIZE, OVERLAP)
print('Chunk size:', CHUNK_SIZE)
print('Overlap:', OVERLAP, f'({OVERLAP/CHUNK_SIZE:.0%})')
print('Number of chunks:', len(chunks))
for i, c in enumerate(chunks[:3], 1):
    print(f'\n--- Chunk {i} ---\n{c}')


Chunk size: 400
Overlap: 60 (15%)
Number of chunks: 6

--- Chunk 1 ---
Sushi is a Japanese dish made of vinegared rice combined with fresh seafood, vegetables, or egg. It is often served with soy sauce, wasabi, and pickled ginger. Nigiri, maki, and sashimi are the most common styles.

Pizza Margherita comes from Naples, Italy. It is topped with tomato sauce, fresh mozzarella, basil, and olive oil. The colors of the toppings are said to represent the Italian flag.

Ta

--- Chunk 2 ---
of the toppings are said to represent the Italian flag.

Tacos are a traditional Mexican food made of a small corn or wheat tortilla folded around a filling. Popular fillings include grilled beef, pork al pastor, chicken, and beans, topped with onion, cilantro, and salsa.

Kabsa is a mixed rice dish that is considered a national dish of Saudi Arabia. It is made with long-grain rice, meat such as

--- Chunk 3 ---
Saudi Arabia. It is made with long-grain rice, meat such as lamb or chicken, and spices like ca

## 3. Convert Chunks into Dense Vectors


In [17]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks, convert_to_numpy=True)
print('Embedding matrix shape:', chunk_embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding matrix shape: (6, 384)


## 4. Semantic Similarity Search
The query is embedded using the exact same model. Cosine similarity is calculated against all chunk vectors.


In [18]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def search(question, top_k=3):
    query_vector = model.encode([question], convert_to_numpy=True)
    scores = cosine_similarity(query_vector, chunk_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    print('Question:', question)
    print('\nTop 3 Answers:')
    for rank, idx in enumerate(top_indices, 1):
        print(f'\nAnswer {rank}')
        print(f'Similarity Score: {scores[idx]:.4f}')
        print(chunks[idx])
    return [(chunks[i], float(scores[i])) for i in top_indices]


## 5. Ask a Question


In [21]:
question = input('Enter your question: ').strip()
if question:
    results = search(question)
else:
    print('Please enter a question.')


Enter your question: What is Kabsa and what ingredients are used to make it?
Question: What is Kabsa and what ingredients are used to make it?

Top 3 Answers:

Answer 1
Similarity Score: 0.5106
of the toppings are said to represent the Italian flag.

Tacos are a traditional Mexican food made of a small corn or wheat tortilla folded around a filling. Popular fillings include grilled beef, pork al pastor, chicken, and beans, topped with onion, cilantro, and salsa.

Kabsa is a mixed rice dish that is considered a national dish of Saudi Arabia. It is made with long-grain rice, meat such as

Answer 2
Similarity Score: 0.4270
n gives it its color, and it may include rabbit, chicken, seafood, and green beans.

Injera is a sour, spongy flatbread made from teff flour, central to Ethiopian and Eritrean cuisine. It is used as both plate and utensil: stews called wat are served on top and eaten by tearing off pieces of injera.

Answer 3
Similarity Score: 0.3516
Saudi Arabia. It is made with long-g